# Notebook 04 : modele final avec optimisation des hyperparametres

**Projet** : Laplace Immo, simulateur d'estimation de prix de maisons  
**Auteurs** : Astou DIALLO, Oumane SALL (EPT, DIC3)  
**Date** : juin 2026

## Objectif

Optimiser les hyperparametres du modele `CatBoost` (identifie comme le meilleur lors des essais) via `RandomizedSearchCV`, puis sauvegarder le pipeline complet (preprocessing + modele) comme artefact serialisable.

## Plan

1. Setup et chargement des donnees
2. Pipeline complet (preprocessing + CatBoost)
3. Definition de l'espace de recherche des hyperparametres
4. RandomizedSearchCV (30 essais) avec tracking MLflow
5. Evaluation du meilleur modele sur le test
6. Comparaison avec CatBoost par defaut
7. Sauvegarde du modele final
8. Synthese

## 1. Setup et chargement

On reutilise les modules de chargement et preprocessing deja construits. Le tracking MLflow utilise une experience dediee `laplace_immo_modele_final` pour separer cette optimisation des essais initiaux.

In [1]:
import sys
import time
import warnings
from pathlib import Path

chemin_projet = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(chemin_projet) not in sys.path:
    sys.path.insert(0, str(chemin_projet))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline

from catboost import CatBoostRegressor

from src.data import charger_donnees, NOM_CIBLE
from src.preprocessing import nettoyer_donnees
from src.features import creer_features
from src.pipeline import construire_pipeline

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

mlflow.set_tracking_uri(f"file:{chemin_projet / 'mlruns'}")
EXPERIMENT_NAME = "laplace_immo_modele_final"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Racine projet : {chemin_projet}")
print(f"Experiment    : {EXPERIMENT_NAME}")

Racine projet : c:\Users\HP\Desktop\DIC3\MLOPS\projet_mlops
Experiment    : laplace_immo_modele_final


In [2]:
# Chargement et preparation deterministe (load + clean + features)
df = charger_donnees()
df = nettoyer_donnees(df)
df = creer_features(df)

X = df.drop(columns=[NOM_CIBLE])
y = np.log1p(df[NOM_CIBLE])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Shape final : {df.shape}")
print(f"X_train : {X_train.shape}, X_test : {X_test.shape}")

Shape final : (1458, 87)
X_train : (1166, 86), X_test : (292, 86)


## 2. Pipeline complet

On utilise `construire_pipeline` pour la partie preprocessing (LotFrontageImputer + ColumnTransformer numerique/ordinal/nominal), enchainee avec `CatBoostRegressor`. C'est ce pipeline qui sera sauvegarde au final.

In [3]:
# Construction du pipeline complet : preprocessing + CatBoost
pipeline = Pipeline([
    ("preprocessing", construire_pipeline(X_train)),
    ("modele", CatBoostRegressor(
        random_state=42,
        verbose=False,
        allow_writing_files=False,
    )),
])

print("Pipeline construit :")
for nom_etape, _ in pipeline.steps:
    print(f"  - {nom_etape}")

Pipeline construit :
  - preprocessing
  - modele


## 3. Espace de recherche

On cible les 4 hyperparametres les plus impactants pour CatBoost :

- `iterations` : nombre d'arbres de boosting
- `depth` : profondeur des arbres (controle la complexite)
- `learning_rate` : taille du pas de gradient (compromis vitesse / convergence)
- `l2_leaf_reg` : regularisation L2 (anti-overfit)

L'espace contient `3 x 5 x 4 x 4 = 240 combinaisons`. `RandomizedSearchCV` en echantillonne 30 au hasard. Avec une CV 5-fold, cela represente 150 entrainements (~ 15 a 20 minutes).

In [4]:
# Distribution des hyperparametres (echantillonnage par RandomizedSearch)
# Prefixe modele__ : indique a sklearn que ces params s'appliquent a l'etape "modele" du pipeline
distributions = {
    "modele__iterations":    [500, 800, 1200],
    "modele__depth":         [5, 6, 7, 8, 9],
    "modele__learning_rate": [0.03, 0.05, 0.08, 0.1],
    "modele__l2_leaf_reg":   [1, 3, 5, 7],
}

nb_combinaisons = 1
for v in distributions.values():
    nb_combinaisons *= len(v)

print(f"Espace de recherche : {nb_combinaisons} combinaisons possibles")
print(f"Echantillonnage : 30 essais (RandomizedSearchCV, random_state=42)")

Espace de recherche : 240 combinaisons possibles
Echantillonnage : 30 essais (RandomizedSearchCV, random_state=42)


## 4. Recherche des hyperparametres optimaux

`RandomizedSearchCV` effectue une cross-validation 5-fold pour chacune des 30 combinaisons echantillonnees, en utilisant le RMSE log comme metrique (alignee avec la metrique Kaggle officielle).

Tout le processus est trace dans un unique run MLflow `CatBoost_Tuned` (espace de recherche + meilleurs hyperparametres + scores).

In [5]:
# RandomizedSearchCV avec tracking MLflow
with mlflow.start_run(run_name="CatBoost_Tuned"):
    recherche = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=distributions,
        n_iter=30,
        cv=5,
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )

    debut = time.time()
    recherche.fit(X_train, y_train)
    duree_recherche = time.time() - debut

    meilleurs_params = recherche.best_params_
    cv_rmse_log_best = float(-recherche.best_score_)

    mlflow.log_param("modele_classe", "CatBoostRegressor")
    mlflow.log_param("methode", "RandomizedSearchCV")
    mlflow.log_param("n_iter", 30)
    mlflow.log_param("cv_folds", 5)
    for cle, valeur in meilleurs_params.items():
        cle_courte = cle.replace("modele__", "best_")
        mlflow.log_param(cle_courte, valeur)

    mlflow.log_metric("cv_rmse_log_best", cv_rmse_log_best)
    mlflow.log_metric("duree_recherche_secondes", float(duree_recherche))

    print(f"\nDuree de la recherche : {duree_recherche:.0f} s ({duree_recherche/60:.1f} min)")
    print(f"Meilleur CV RMSE log : {cv_rmse_log_best:.4f}")
    print(f"\nMeilleurs hyperparametres :")
    for cle, valeur in meilleurs_params.items():
        print(f"  {cle.replace('modele__', '')} : {valeur}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Duree de la recherche : 1416 s (23.6 min)
Meilleur CV RMSE log : 0.1163

Meilleurs hyperparametres :
  learning_rate : 0.05
  l2_leaf_reg : 7
  iterations : 1200
  depth : 5


In [6]:
# Top 5 des combinaisons testees, triees par CV RMSE log
resultats_cv = pd.DataFrame(recherche.cv_results_)
colonnes_interet = [
    "param_modele__iterations",
    "param_modele__depth",
    "param_modele__learning_rate",
    "param_modele__l2_leaf_reg",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]
top5 = resultats_cv[colonnes_interet].sort_values("rank_test_score").head(5).copy()
top5["cv_rmse_log"] = (-top5["mean_test_score"]).round(4)
top5["cv_std"] = top5["std_test_score"].round(4)
top5 = top5.drop(columns=["mean_test_score", "std_test_score"])
top5.columns = ["iterations", "depth", "learning_rate", "l2_leaf_reg", "rang", "cv_rmse_log", "cv_std"]
top5

,iterations,depth,learning_rate,l2_leaf_reg,rang,cv_rmse_log,cv_std
10,1200,5,0.05,7,1,0.1163,0.0082
0,800,5,0.03,5,2,0.1166,0.0094
8,500,5,0.05,5,3,0.1171,0.0083
2,1200,6,0.05,7,4,0.1179,0.0085
23,800,5,0.08,7,5,0.1179,0.0089


## 5. Evaluation sur le test

Le meilleur pipeline trouve par la recherche est evalue sur le test set jamais vu. On reporte les metriques en log (Kaggle) et en dollars (lisibilite metier).

In [7]:
pipeline_tune = recherche.best_estimator_
y_pred_log = pipeline_tune.predict(X_test)

test_rmse_log = float(np.sqrt(mean_squared_error(y_test, y_pred_log)))
test_r2 = float(r2_score(y_test, y_pred_log))
y_test_dollars = np.expm1(y_test)
y_pred_dollars = np.expm1(y_pred_log)
test_rmse_dollars = float(np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars)))
test_mae_dollars = float(mean_absolute_error(y_test_dollars, y_pred_dollars))

with mlflow.start_run(run_name="CatBoost_Tuned_eval"):
    for cle, valeur in meilleurs_params.items():
        mlflow.log_param(cle, valeur)
    mlflow.log_metric("test_rmse_log", test_rmse_log)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("test_rmse_dollars", test_rmse_dollars)
    mlflow.log_metric("test_mae_dollars", test_mae_dollars)
    mlflow.sklearn.log_model(pipeline_tune, artifact_path="model")

print("=== CatBoost tune ===")
print(f"  Test RMSE log : {test_rmse_log:.4f}")
print(f"  Test R2       : {test_r2:.4f}")
print(f"  Test RMSE $   : {test_rmse_dollars:>10,.0f}")
print(f"  Test MAE  $   : {test_mae_dollars:>10,.0f}")

2026/06/08 10:04:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


=== CatBoost tune ===
  Test RMSE log : 0.1151
  Test R2       : 0.9214
  Test RMSE $   :     19,003
  Test MAE  $   :     13,583


## 6. Comparaison avec CatBoost par defaut

On entraine un CatBoost avec les hyperparametres par defaut (`iterations=500, learning_rate=0.05, depth=5` comme dans les essais initiaux) pour mesurer le gain apporte par le tuning.

In [8]:
# Re-entrainement de CatBoost avec les hyperparametres initiaux (essais notebook 03)
pipeline_default = Pipeline([
    ("preprocessing", construire_pipeline(X_train)),
    ("modele", CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=5,
        random_state=42, verbose=False, allow_writing_files=False,
    )),
])
pipeline_default.fit(X_train, y_train)
y_pred_log_def = pipeline_default.predict(X_test)

test_rmse_log_def = float(np.sqrt(mean_squared_error(y_test, y_pred_log_def)))
test_r2_def = float(r2_score(y_test, y_pred_log_def))
test_rmse_dollars_def = float(np.sqrt(mean_squared_error(y_test_dollars, np.expm1(y_pred_log_def))))
test_mae_dollars_def = float(mean_absolute_error(y_test_dollars, np.expm1(y_pred_log_def)))

comparaison = pd.DataFrame({
    "Metrique": ["Test RMSE log", "Test R2", "Test RMSE dollars", "Test MAE dollars"],
    "CatBoost defaut": [
        round(test_rmse_log_def, 4),
        round(test_r2_def, 4),
        int(test_rmse_dollars_def),
        int(test_mae_dollars_def),
    ],
    "CatBoost tune": [
        round(test_rmse_log, 4),
        round(test_r2, 4),
        int(test_rmse_dollars),
        int(test_mae_dollars),
    ],
})
comparaison["Gain"] = comparaison["CatBoost defaut"] - comparaison["CatBoost tune"]
comparaison

,Metrique,CatBoost defaut,CatBoost tune,Gain
0,Test RMSE log,0.1153,0.1151,0.0002
1,Test R2,0.9211,0.9214,-0.0003
2,Test RMSE dollars,18734.0000,19003.0000,-269.0000
3,Test MAE dollars,13638.0000,13582.0000,56.0000


## 7. Sauvegarde du modele final

Le pipeline complet (preprocessing + CatBoost tune) est serialise dans `models/final_model.pkl` via `joblib`. Ce fichier sera utilise par le pipeline DVC, l'API FastAPI et l'interface Streamlit pour produire des predictions sur de nouvelles maisons.

In [9]:
chemin_modele = chemin_projet / "models" / "final_model.pkl"
chemin_modele.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline_tune, chemin_modele)

taille_mo = chemin_modele.stat().st_size / (1024 * 1024)
print(f"Modele sauvegarde : {chemin_modele}")
print(f"Taille : {taille_mo:.2f} Mo")

Modele sauvegarde : c:\Users\HP\Desktop\DIC3\MLOPS\projet_mlops\models\final_model.pkl
Taille : 0.72 Mo


In [10]:
# Verification : on recharge le modele et on verifie qu'il produit les memes predictions
pipeline_recharge = joblib.load(chemin_modele)
y_pred_recharge = pipeline_recharge.predict(X_test)

ecart_max = float(np.abs(y_pred_recharge - y_pred_log).max())
print(f"Ecart maximum entre predictions originales et apres rechargement : {ecart_max:.2e}")
print("Modele recharge identique au modele d'origine" if ecart_max < 1e-10 else "Difference detectee")

Ecart maximum entre predictions originales et apres rechargement : 0.00e+00
Modele recharge identique au modele d'origine


## 8. Synthese

Le modele final est un `CatBoostRegressor` avec les hyperparametres optimises par `RandomizedSearchCV`, integre dans un Pipeline complet de preprocessing.

Le pipeline complet est serialise dans `models/final_model.pkl` et peut etre rechargee identiquement pour la production.